# Burel — A/B nested-vs-vanilla (a parità di parametri)

Addestra il **Transformer vanilla** (~20.4M param, contro Burel ~20.17M) sugli STESSI dati, poi esegue il confronto a tre curve **Burel-ON / Burel-OFF / Vanilla** sulle stesse finestre.

**Prima:** metti su Drive `burel_vanilla_ab.zip` e questo notebook. Poi Runtime → *Change runtime type* → **GPU (T4)** ed esegui le celle in ordine.

L'addestramento del vanilla dura ~1–1.5h su T4 (niente 2° ordine, è veloce). È **resumabile**: se Colab si disconnette, ri-esegui la cella 6 e riparte da `vanilla_last.pt`.

In [ ]:
# Cella 1 — monta Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Cella 2 — trova zip, checkpoint Burel e cache BPE su Drive (override manuale se serve)
import glob, os
DRIVE = '/content/drive/MyDrive'
ZIP_PATH = CKPT_BUREL = DATA_DIR = None   # lasciali None per la ricerca automatica

if ZIP_PATH is None:
    h = sorted(glob.glob(f'{DRIVE}/**/burel_vanilla_ab.zip', recursive=True)); print('zip:', h); ZIP_PATH = h[0] if h else None
if CKPT_BUREL is None:
    h = sorted(glob.glob(f'{DRIVE}/**/burel_best.pt', recursive=True)); print('burel_best:', h); CKPT_BUREL = h[0] if h else None
if DATA_DIR is None:
    metas = sorted(glob.glob(f'{DRIVE}/**/meta.pkl', recursive=True))
    cand = [os.path.dirname(m) for m in metas
            if os.path.exists(os.path.join(os.path.dirname(m),'val.bin'))
            and os.path.exists(os.path.join(os.path.dirname(m),'tokenizer.json'))]
    print('cache BPE:', cand); DATA_DIR = cand[0] if cand else None

print('\n=> ZIP_PATH   =', ZIP_PATH)
print('=> CKPT_BUREL =', CKPT_BUREL)
print('=> DATA_DIR   =', DATA_DIR)
assert ZIP_PATH and CKPT_BUREL and DATA_DIR, 'Manca qualcosa: imposta i path a mano qui sopra.'

In [ ]:
# Cella 3 — scompatta, installa tokenizer, ripristina la cache BPE, allinea la cartella di backup
import zipfile, shutil, os, pickle, yaml
ROOT = '/content/burel_vanilla_ab'
if os.path.exists(ROOT): shutil.rmtree(ROOT)
with zipfile.ZipFile(ZIP_PATH) as z: z.extractall('/content')
print('codice in', ROOT, '->', os.listdir(ROOT))
!pip -q install tokenizers

# cache BPE (la stessa di Burel) -> data_cache/
CACHE = os.path.join(ROOT, 'data_cache'); os.makedirs(CACHE, exist_ok=True)
for fn in ('meta.pkl','val.bin','train.bin','tokenizer.json'):
    src = os.path.join(DATA_DIR, fn)
    if os.path.exists(src): shutil.copy2(src, os.path.join(CACHE, fn)); print('copiato', fn)
    elif fn != 'train.bin': raise FileNotFoundError(f'manca {fn} in {DATA_DIR}')
meta = pickle.load(open(os.path.join(CACHE,'meta.pkl'),'rb'))
assert meta.get('encoding')=='bpe', 'cache NON BPE: controlla DATA_DIR'
print('vocab:', meta['vocab_size'], 'encoding:', meta['encoding'])

# salva vanilla_best.pt nella STESSA cartella di burel_best.pt
cfgp = os.path.join(ROOT,'study_vanilla','config.yaml'); cfg = yaml.safe_load(open(cfgp))
cfg['train']['drive_backup_dir'] = os.path.dirname(CKPT_BUREL)
yaml.safe_dump(cfg, open(cfgp,'w'), sort_keys=False)
print('vanilla -> backup in', cfg['train']['drive_backup_dir'])
print('max_iters', cfg['train']['max_iters'], '| batch', cfg['train']['batch_size'], '| lr', cfg['train']['learning_rate'])

In [ ]:
# Cella 4 — sanity: conta i parametri del vanilla (deve essere ~20.4M, vicino a Burel ~20.17M)
import sys; sys.path.insert(0, ROOT)
from study_vanilla.model import VanillaGPT, count_parameters
import yaml
mc = yaml.safe_load(open(os.path.join(ROOT,'study_vanilla','config.yaml')))['model']
m = VanillaGPT(vocab_size=meta['vocab_size'], **{k:mc[k] for k in ('d_model','n_head','n_layer','d_ff','context','dropout','activation','tie_weights')})
print(f'VanillaGPT: {count_parameters(m):,} parametri  (Burel ~20,170,000)')
del m

In [ ]:
# Cella 5 — costruisci il dominio MAI VISTO = codice Python (il sorgente stesso)
import glob
DOMAIN = os.path.join(ROOT,'domain_code.txt')
py = sorted(glob.glob(os.path.join(ROOT,'burel','**','*.py'),recursive=True)) + \
     sorted(glob.glob(os.path.join(ROOT,'study_vanilla','*.py'))) + \
     sorted(glob.glob(os.path.join(ROOT,'scripts','*.py')))
open(DOMAIN,'w').write('\n'.join(open(p).read() for p in py))
print(f'{len(py)} file -> {DOMAIN} ({os.path.getsize(DOMAIN)} byte)')

In [ ]:
# Cella 6 — ADDESTRA IL VANILLA (~1-1.5h su T4). Resumabile: se si disconnette, ri-esegui.
!cd "{ROOT}" && python study_vanilla/train.py

In [ ]:
# Cella 7 — A/B PERNO: Burel-ON / Burel-OFF / Vanilla sulle stesse finestre
VANILLA_CKPT = f'{ROOT}/checkpoints_vanilla/vanilla_best.pt'
!cd "{ROOT}" && python scripts/compare_three.py --burel_ckpt "{CKPT_BUREL}" --vanilla_ckpt "{VANILLA_CKPT}" --domain_file "{DOMAIN}" --windows 128 --batch 8

## Come leggere
- **HEADLINE best val**: stesso vocab/dati → confronto diretto. Chi ha la val più bassa vince a parità di parametri.
- **loss-per-chunk** su CONTROL e UNSEEN: colonna `ON-Van`. Se è **negativa** (Burel-ON sotto il vanilla) la memoria si ripaga; se è ~0 o positiva, l'attenzione piena del vanilla fa già tutto e la mossa onesta è scalare il vanilla.

Lo script stampa un **VERDETTO A/B**. Incolla qui tutto l'output e lo leggiamo insieme.